In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from pyproj import Transformer
from sklearn.neighbors import KernelDensity

# --------------------------------------------------------------------------
# Parámetros de configuración (reemplazan a la línea de comandos)
# --------------------------------------------------------------------------
CSV_PATH = "incidencia_C5_periAB_2018_2023.csv"
OUT_DIR = Path("./input3")
# Si quieres filtrar por clases, pon las clases aquí. Si quieres TODO, pon None o lista vacía.
CLASES_DELITO = []
BANDWIDTH = 200.0
CELL_SIZE = 5.0
BUFFER = 200.0

OUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------
# 1. Carga y filtrado
# --------------------------------------------------------------------------
print("Cargando y filtrando datos...")
df = pd.read_csv(CSV_PATH, encoding="utf-8", low_memory=False)

if CLASES_DELITO:
    antes = len(df)
    df = df[df["clas_con_f_alarma"].isin(CLASES_DELITO)].copy()
    print(f"Filtro clas_con_f_alarma in {CLASES_DELITO}: {antes} -> {len(df)} filas")

df = df.dropna(subset=["longitud", "latitud"])
df = df[(df["longitud"].between(-100, -98)) & (df["latitud"].between(19, 20))]

df["fecha_delito"] = pd.to_datetime(df["fecha_delito"], errors="coerce")
df = df.dropna(subset=["fecha_delito"])
df["year"] = df["fecha_delito"].dt.year
df["month"] = df["fecha_delito"].dt.month

# --------------------------------------------------------------------------
# 2. Reproyección a UTM 14N
# --------------------------------------------------------------------------
print("Reproyectando coordenadas a UTM 14N...")
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32614", always_xy=True)
x, y = transformer.transform(df["longitud"].values, df["latitud"].values)
df["x"], df["y"] = x, y

# --------------------------------------------------------------------------
# 3. Extent fijo y Grid
# --------------------------------------------------------------------------
print("Calculando extent fijo y malla...")
minx, maxx = df["x"].min() - BUFFER, df["x"].max() + BUFFER
miny, maxy = df["y"].min() - BUFFER, df["y"].max() + BUFFER
extent = (minx, miny, maxx, maxy)

n_cols = int(np.ceil((maxx - minx) / CELL_SIZE))
n_rows = int(np.ceil((maxy - miny) / CELL_SIZE))
xs = minx + (np.arange(n_cols) + 0.5) * CELL_SIZE
ys = maxy - (np.arange(n_rows) + 0.5) * CELL_SIZE
xx, yy = np.meshgrid(xs, ys)
grid_points = np.column_stack([xx.ravel(), yy.ravel()])
transform = from_origin(minx, maxy, CELL_SIZE, CELL_SIZE)
grid_shape = (n_rows, n_cols)

print(f"Extent: {extent}")
print(f"Grid shape: {grid_shape} (filas, columnas) | celda: {CELL_SIZE}m x {CELL_SIZE}m")

# --------------------------------------------------------------------------
# 4. Bucle KDE por mes (con salida inmediata en consola)
# --------------------------------------------------------------------------
years = range(2018, 2024)
months = range(1, 13)
total = len(list(years)) * 12
i = 0

print("\nIniciando generación de rasters mensuales...")
for year in years:
    for month in months:
        i += 1
        sub = df[(df["year"] == year) & (df["month"] == month)]
        puntos_xy = sub[["x", "y"]].to_numpy()

        if len(puntos_xy) > 0:
            kde = KernelDensity(kernel="gaussian", bandwidth=BANDWIDTH)
            kde.fit(puntos_xy)
            log_dens = kde.score_samples(grid_points)
            dens = np.exp(log_dens).astype(np.float32).reshape(grid_shape)
        else:
            dens = np.zeros(grid_shape, dtype=np.float32)

        fname = OUT_DIR / f"kde_{int(BANDWIDTH)}_{CELL_SIZE:.0f}m_{year}_{month}.tif"
        
        with rasterio.open(
            fname, "w",
            driver="GTiff",
            height=dens.shape[0],
            width=dens.shape[1],
            count=1,
            dtype="float32",
            crs="EPSG:32614",
            transform=transform,
            nodata=None,
        ) as dst:
            dst.write(dens, 1)

        # El parámetro flush=True fuerza a que Jupyter imprima la línea en tiempo real
        print(f"[{i}/{total}] {fname.name}  (n_puntos={len(sub)}, max_densidad={dens.max():.6f})", flush=True)

print("\n¡Proceso finalizado con éxito!")

Cargando y filtrando datos...
Reproyectando coordenadas a UTM 14N...
Calculando extent fijo y malla...
Extent: (np.float64(483380.31571371), np.float64(2146974.9443344753), np.float64(488633.3782070932), np.float64(2150420.135134353))
Grid shape: (690, 1051) (filas, columnas) | celda: 5.0m x 5.0m

Iniciando generación de rasters mensuales...
[1/72] kde_200_5m_2018_1.tif  (n_puntos=735, max_densidad=0.000000)
[2/72] kde_200_5m_2018_2.tif  (n_puntos=728, max_densidad=0.000000)
[3/72] kde_200_5m_2018_3.tif  (n_puntos=918, max_densidad=0.000000)
[4/72] kde_200_5m_2018_4.tif  (n_puntos=795, max_densidad=0.000000)
[5/72] kde_200_5m_2018_5.tif  (n_puntos=848, max_densidad=0.000000)
[6/72] kde_200_5m_2018_6.tif  (n_puntos=741, max_densidad=0.000000)
[7/72] kde_200_5m_2018_7.tif  (n_puntos=768, max_densidad=0.000000)
[8/72] kde_200_5m_2018_8.tif  (n_puntos=865, max_densidad=0.000000)
[9/72] kde_200_5m_2018_9.tif  (n_puntos=768, max_densidad=0.000000)
[10/72] kde_200_5m_2018_10.tif  (n_puntos=88